Two event breakdown figures requested by R2, CEE.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# fire observations
fire_obs = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/FRYv2.0_FireCCI51_6D_2001-2020_study_area_fire_observations_w_region.shp")

In [ ]:
#calculate unnormalized seasonality
fire_seas = pd.DataFrame(columns = ["Region", "Month", "EV", "BA"]) #region, month, number of fire event, cumulative burned area
region_list = ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for reg in region_list:
    
    fire_obs_reg = fire_obs[fire_obs["region"] == reg].copy()
    fire_obs_reg["start_date"] = pd.to_datetime(fire_obs_reg["start_date"])
    fire_seas_reg = pd.DataFrame(columns = ["Region", "Month", "EV", "BA"])
    fire_seas_reg["Region"] = [reg for _ in range(12)]
    fire_seas_reg["Month"] = months
    
    for ind, mon in enumerate(months):
        
        fire_obs_reg_mon = fire_obs_reg[fire_obs_reg['start_date'].dt.month == ind+1]
        
        #avg number of events across the 20 yrs period
        EV = len(fire_obs_reg_mon)/20
        
        #avg cumulative burned area across the 20 yrs period
        BA = fire_obs_reg_mon["area"].sum()/20

        fire_seas_reg.loc[fire_seas_reg["Month"] == mon, "EV"] = EV
        fire_seas_reg.loc[fire_seas_reg["Month"] == mon, "BA"] = BA


    fire_seas = pd.concat([fire_seas, fire_seas_reg], ignore_index = True)

In [ ]:
region_lb = ["(a) BI", "(b) IP", "(c) FR", "(d) CE", "(e) AL", "(f) SEE", "(g) NEE", "(h) SC", "(i) WMD", "(j) EMD"]

#visualization
fig, axes = plt.subplots(5,2, figsize = (15, 13))
axes = axes.flatten()

for reg, ax in zip(region_list, axes):
    
    fire_seas_reg = fire_seas[fire_seas["Region"] == reg]

    
    # Plot Variable1 as bars
    ax.bar(np.arange(0.5, 12.5, 1), fire_seas_reg['BA']*100, color='#6c3baa', width = 0.6) #convert km2 to hectare
    ax.set_xlabel('')
    ax.set_xticks(np.arange(0.5, 12.5, 1))
    ax.set_xticklabels(months, fontsize = 12)
    ax.set_ylabel('Burned area (ha)', color='#6c3baa', fontsize = 13)
    ax.tick_params(axis='y', labelcolor='#6c3baa')
    ax.text(0.01, 0.85, f'{region_lb[region_list.index(reg)]}', fontsize = 15, transform = ax.transAxes)
    
    # Create a secondary y-axis
    ax2 = ax.twinx()
    
    # Plot Variable2 as dots
    ax2.plot(np.arange(0.5, 12.5, 1), fire_seas_reg['EV'], color='#e60000', marker='o')
    ax2.set_ylabel('Event number (-)', color='#e60000', fontsize = 13)
    ax2.tick_params(axis='y', labelcolor='#e60000')
    ax2.set_ylim(bottom=0)


plt.tight_layout()

plt.savefig('/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_unnormalized_fire_seasonality.png', dpi = 600, bbox_inches = 'tight')